In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
products = [
    {
        "model": "55U7KQ",
        "category": "TV",
        "brand": "Hisense",
        "description": "55 inch 4K Mini LED Smart TV, VIDAA, 144Hz"
    },
    {
        "model": "55U6KQ",
        "category": "TV",
        "brand": "Hisense",
        "description": "55 inch 4K Mini LED Smart TV, VIDAA"
    },
    {
        "model": "55A6K",
        "category": "TV",
        "brand": "Hisense",
        "description": "55 inch 4K LED Smart TV, VIDAA"
    },
    {
        "model": "65U7KQ",
        "category": "TV",
        "brand": "Hisense",
        "description": "65 inch 4K Mini LED Smart TV, VIDAA, 144Hz"
    },
    {
        "model": "50A6K",
        "category": "TV",
        "brand": "Hisense",
        "description": "50 inch 4K LED Smart TV, VIDAA"
    }
]

df = pd.DataFrame(products)

In [3]:
df

,model,category,brand,description
0,55U7KQ,TV,Hisense,"55 inch 4K Mini LED Smart TV, VIDAA, 144Hz"
1,55U6KQ,TV,Hisense,"55 inch 4K Mini LED Smart TV, VIDAA"
2,55A6K,TV,Hisense,"55 inch 4K LED Smart TV, VIDAA"
3,65U7KQ,TV,Hisense,"65 inch 4K Mini LED Smart TV, VIDAA, 144Hz"
4,50A6K,TV,Hisense,"50 inch 4K LED Smart TV, VIDAA"


In [4]:
df["search_text"] = (
    df["brand"] + " " +
    df["model"] + " " +
    df["category"] + " " +
    df["description"]
).str.lower()

df[["model", "search_text"]]

,model,search_text
0,55U7KQ,hisense 55u7kq tv 55 inch 4k mini led smart tv...
1,55U6KQ,hisense 55u6kq tv 55 inch 4k mini led smart tv...
2,55A6K,"hisense 55a6k tv 55 inch 4k led smart tv, vidaa"
3,65U7KQ,hisense 65u7kq tv 65 inch 4k mini led smart tv...
4,50A6K,"hisense 50a6k tv 50 inch 4k led smart tv, vidaa"


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
vectorizer = TfidfVectorizer(analyzer='char',ngram_range=(2,4),lowercase=True)
X = vectorizer.fit_transform(df['search_text'])

In [8]:
print("TF-IDF Matrix:", X.shape)

TF-IDF Matrix: (5, 221)


In [9]:
def search_hisense(query, top_k=3):
    query = str(query).strip()

    if not query:
        return pd.DataFrame()

    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(
        query_vector, X
    ).flatten()

    indices = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[indices].copy()
    results["similarity"] = similarities[indices]

    return results[
        ["model", "category", "description", "similarity"]
    ].reset_index(drop=True)


In [10]:
search_hisense("Hisense 55U7KQ")

,model,category,description,similarity
0,55U7KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA, 144Hz",0.508324
1,55U6KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA",0.362480
2,55A6K,TV,"55 inch 4K LED Smart TV, VIDAA",0.306659


In [11]:
display(search_hisense("55U7K"))
display(search_hisense("Mini LED 65 inch"))
display(search_hisense("Hisense 4K TV"))

,model,category,description,similarity
0,55U7KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA, 144Hz",0.318772
1,65U7KQ,TV,"65 inch 4K Mini LED Smart TV, VIDAA, 144Hz",0.172306
2,55U6KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA",0.111946


,model,category,description,similarity
0,65U7KQ,TV,"65 inch 4K Mini LED Smart TV, VIDAA, 144Hz",0.539805
1,55U6KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA",0.352625
2,55U7KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA, 144Hz",0.320695


,model,category,description,similarity
0,55A6K,TV,"55 inch 4K LED Smart TV, VIDAA",0.455566
1,50A6K,TV,"50 inch 4K LED Smart TV, VIDAA",0.416734
2,55U6KQ,TV,"55 inch 4K Mini LED Smart TV, VIDAA",0.349463


In [12]:

while True:
    query = input("Enter Hisense model (exit to stop): ").strip()

    if query.lower() == "exit":
        break

    if not query:
        print("Please enter a model.")
        continue

    results = search_hisense(query, top_k=1)

    if results.empty or results.iloc[0]["similarity"] < 0.15:
        print("Product not found.")
        continue

    product = results.iloc[0]

    print("\nHISENSE PRODUCT INFORMATION")
    print("--------------------------")
    print("Model:", product["model"])
    print("Category:", product["category"])
    print("Description:", product["description"])
    print("Similarity:", round(product["similarity"] * 100, 2), "%")
    print()


Enter Hisense model (exit to stop):  Hisense 55U7KQ



HISENSE PRODUCT INFORMATION
--------------------------
Model: 55U7KQ
Category: TV
Description: 55 inch 4K Mini LED Smart TV, VIDAA, 144Hz
Similarity: 50.83 %



Enter Hisense model (exit to stop):  ok


Product not found.


Enter Hisense model (exit to stop):  exit


In [13]:

import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(X, "tfidf_matrix.pkl")

df.to_csv("hisense_products.csv", index=False)

print("Model and dataset saved successfully!")


Model and dataset saved successfully!
